## Concept focus — Context managers and resource lifecycles

Context managers encode setup and teardown reliably, even in failing code. They are the control structure behind safe file usage, transaction scopes, temporary state, and dynamic cleanup stacks.

```text
enter context --> do work --> exit context
      |                        |
      v                        v
   acquire                  release / cleanup

with resource():
    use_resource()
```

### How to think about it
The key question is not “can I open this resource?” but “how do I guarantee cleanup under success, error, and interruption?” `contextlib` exists to make that guarantee composable and explicit.

### Visual references and further study
- [contextlib documentation](https://docs.python.org/3/library/contextlib.html)
- [Python docs — with statement](https://docs.python.org/3/reference/compound_stmts.html#the-with-statement)
- [Real Python — context managers](https://realpython.com/python-with-statement/)
- [Python Tutor visualizer](https://pythontutor.com/visualize.html)

---

# Module 15 — Decorators, Closures, and functools

## Exercise 15.4 — contextlib, including the case hand-rolling gets wrong

Run:  python ex04_contextlib.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `functools.wraps` is not optional

In [ ]:
@log_calls
def add(a, b):
    """Add two numbers."""

add.__name__      # 'wrapper'    <- wrong
add.__doc__       # None         <- gone
inspect.signature(add)   # (*args, **kwargs)   <- useless

The wrapper replaced the function, so all of its metadata is the wrapper's.
What breaks, concretely:

- `help()` and every documentation generator
- debuggers and profilers reporting "wrapper" for every decorated function
- **pytest fixture resolution**, which inspects parameter names
- **FastAPI and Pydantic**, which build schemas from signatures
- `singledispatch`, which reads annotations
- any logging that uses `__name__`

In [ ]:
import functools

def log_calls(fn):
    @functools.wraps(fn)          # copies __name__, __doc__, __module__,
    def wrapper(*args, **kwargs): # __qualname__, __dict__, and sets __wrapped__
        return fn(*args, **kwargs)
    return wrapper

`__wrapped__` is what lets `inspect.signature` see through the wrapper to the
real signature. **Always use `wraps`.** There is no case where omitting it is
correct.

---

## Concept 3. Decorators with arguments: three levels

In [ ]:
def retry(attempts=3, delay=1.0):        # 1. the FACTORY takes the arguments
    def decorator(fn):                   # 2. the DECORATOR takes the function
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):    # 3. the WRAPPER takes the call
            for attempt in range(attempts):
                try:
                    return fn(*args, **kwargs)
                except Exception:
                    if attempt == attempts - 1:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(attempts=5)          # note: CALLED. retry(5) returns `decorator`.
def flaky(): ...

`@retry` without parentheses passes the *function* as `attempts`, and the error
appears far away and makes no sense. To support both forms:

```text
def retry(fn=None, *, attempts=3):
    if fn is None:                       # called with arguments
        return functools.partial(retry, attempts=attempts)
    @functools.wraps(fn)
    def wrapper(*a, **kw): ...
    return wrapper

@retry              # works
@retry(attempts=5)  # also works
```


---

## Concept 5. `functools`

### `lru_cache` / `cache`

In [ ]:
@functools.lru_cache(maxsize=128)
def expensive(n: int) -> int: ...

@functools.cache                  # 3.9+: unbounded lru_cache
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

expensive.cache_info()            # hits, misses, maxsize, currsize
expensive.cache_clear()

Five things to know before using it:

1. **Arguments must be hashable.** A list argument raises `TypeError`.
2. **Equal-but-distinct arguments can collide.** `1`, `1.0` and `True` are equal
   and hash equally (Module 03), so a function that treats them differently can
   get the wrong cached answer. The exact behaviour is subtler than it looks —
   `lru_cache` has a fast path for a single `int` or `str` argument, so the real
   grouping is not the one you would predict. Exercise 15.3 measures it. The
   safe rule: if your function's behaviour depends on the *type* of a numeric
   argument, do not cache it by that argument.
3. **`f(1)` and `f(x=1)` are different entries.** Same call, two cache slots.
4. **It keeps a strong reference to every argument and result.** `@cache` on a
   method keeps every instance alive forever — a genuine and common memory leak.
   Use `maxsize`, or `cached_property`, or a `WeakValueDictionary`.
5. **Only cache pure functions.** A cached function with side effects performs
   them once and silently skips them thereafter.

### `partial`

In [ ]:
from functools import partial
int2 = partial(int, base=2)
int2("1010")                       # 10
sorted(rows, key=partial(get_field, "name"))

`partial` beats a lambda for a callback: it has a useful `repr`, it is
picklable (so it works with `multiprocessing`, Module 21), and it does not
capture variables by reference — which sidesteps Module 04's late-binding trap.

### `singledispatch`

In [ ]:
@functools.singledispatch
def serialise(obj) -> str:
    raise TypeError(f"cannot serialise {type(obj).__name__}")

@serialise.register
def _(obj: datetime) -> str: return obj.isoformat()

@serialise.register
def _(obj: Decimal) -> str: return str(obj)

Type-based dispatch without an `isinstance` chain, and — importantly —
**open for extension**: a third party can register a handler for their own type
without touching your code. This is the Visitor pattern, dissolved (Module 12).

`singledispatchmethod` does the same for methods.

### `cached_property`, `total_ordering`, `reduce`

In [ ]:
@functools.cached_property        # Module 08: computed once, stored in __dict__
def stats(self): ...

@functools.total_ordering         # Module 09: fills in <=, >, >= from < and ==
class Version: ...

functools.reduce(operator.mul, nums, 1)     # rarely clearer than a loop

`reduce` is worth knowing and rarely worth using. `sum`, `math.prod`,
`itertools.accumulate` and an explicit loop are all clearer.

---

## Concept 6. `contextlib`

In [ ]:
from contextlib import contextmanager, suppress, ExitStack, closing, nullcontext

@contextmanager                    # Module 14: __enter__/__exit__ from a generator
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{time.perf_counter() - start:.3f}s")

with suppress(FileNotFoundError):  # the ONE legitimate exception-swallower
    path.unlink()

with ExitStack() as stack:         # a DYNAMIC number of context managers
    files = [stack.enter_context(open(p)) for p in paths]
    # all closed on exit, in reverse order, even if one open() fails

with nullcontext():                # a no-op, for conditional context managers
    ...

`ExitStack` is the answer to "I need N context managers where N is not known
until run time", and it handles the case where entering the third one raises —
the first two are still unwound correctly. Hand-rolled versions of this are
almost always subtly wrong.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `@` is sugar
- Section 2: `functools.wraps` is not optional
- Section 3: Decorators with arguments: three levels
- Section 4: Stacking order
- Section 5: `functools`
- Section 6: `contextlib`
- Section 7: Class-based decorators, and when to use one

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import contextlib
from pathlib import Path
from typing import Any

# TODO 1  @contextmanager-based `chdir` that restores the previous directory
#         even on an exception. (3.11 has contextlib.chdir -- write it anyway,
#         then compare.)
#
# TODO 2  A `transaction` context manager over a fake connection: commit on
#         success, rollback on exception, and NEVER swallow. Support nesting
#         via savepoints.
#
# TODO 3  open_all(paths) using ExitStack: open N files, yield them, close all
#         of them in reverse order on exit.
#         THE POINT: make open() fail on the THIRD of five paths, and assert
#         that the first two were still closed. Write the hand-rolled version
#         first (a list of files and a try/finally loop) and find the bug in
#         it -- it is subtle and it is why ExitStack exists.
#
# TODO 4  A `maybe` helper returning either a real context manager or
#         contextlib.nullcontext(), so a caller can write:
#             with maybe(verbose, timer()):
#                 ...
#         instead of duplicating the body under an if.

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    original = Path.cwd()
    with chdir("/tmp"):                              # type: ignore[name-defined]
        assert Path.cwd() == Path("/tmp")
    assert Path.cwd() == original

    try:
        with chdir("/tmp"):                          # type: ignore[name-defined]
            raise ValueError("boom")
    except ValueError:
        pass
    assert Path.cwd() == original, "must restore on an exception too"

    conn = FakeConnection()                          # type: ignore[name-defined]
    with transaction(conn):                          # type: ignore[name-defined]
        conn.execute("INSERT 1")
    assert conn.log[-1] == "COMMIT", conn.log

    conn2 = FakeConnection()                         # type: ignore[name-defined]
    try:
        with transaction(conn2):                     # type: ignore[name-defined]
            conn2.execute("INSERT 1")
            raise RuntimeError("fail")
    except RuntimeError:
        pass
    else:
        raise AssertionError("transaction must not swallow")
    assert conn2.log[-1] == "ROLLBACK", conn2.log

    import tempfile
    with tempfile.TemporaryDirectory() as td:
        paths = [Path(td) / f"f{i}.txt" for i in range(5)]
        for p in paths[:2]:
            p.write_text("data", encoding="utf-8")
        # paths[2] does not exist -> open() raises on the third
        opened: list[Any] = []
        try:
            with open_all(paths, record=opened):     # type: ignore[name-defined]
                raise AssertionError("should not reach the body")
        except FileNotFoundError:
            pass
        assert len(opened) == 2, "two files were opened"
        assert all(f.closed for f in opened), (
            "the first two files must be closed despite the third failing"
        )

    print("all contextlib checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.